# LeetCode 542 — 01 Matrix bằng Python

Link đề: https://leetcode.com/problems/01-matrix/ · Độ khó: **Medium** · Chủ đề: Breadth-First Search, Array, Dynamic Programming, Matrix

Skeleton LeetCode:

```python
class Solution:
    def updateMatrix(self, mat: List[List[int]]) -> List[List[int]]:
        ...
```

Ghi chú: bài này giống hệt LeetCode 1765 (*Map of Highest Peak*). Notebook dạy từ "BFS xuất phát từ mỗi số 1" chậm, tới **BFS đa nguồn** (multi-source BFS) từ mọi số 0 cùng lúc, rồi **quy hoạch động hai lượt quét** và **biến đổi khoảng cách tách theo chiều**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: với mỗi ô của ma trận nhị phân, tìm **khoảng cách (số bước đi ngang/dọc) tới số `0` gần nhất**.
2. Nắm các công cụ Python: ma trận là list lồng list (`m, n = len(mat), len(mat[0])`), tạo ma trận đúng cách (`[[-1] * n for _ in range(m)]`), `collections.deque` làm hàng đợi, duyệt 4 hướng với kiểm tra biên.
3. Đi từ **BFS riêng cho từng ô 1** → **BFS đa nguồn** → **DP hai lượt quét** → **tách theo cột rồi theo hàng** → **công thức Manhattan vét cạn** (dùng làm oracle kiểm tra).
4. Hiểu mấu chốt: **đảo chiều tìm kiếm** — thay vì mỗi số 1 đi tìm số 0, cho **tất cả số 0 cùng lan ra** như sóng; ô nào bị sóng chạm tới trước tiên ở bước `d` thì đáp án của nó là `d`.
5. Tránh các lỗi kinh điển: `[[-1] * n] * m` (các hàng dùng chung một list), quên gán `0` và đưa các số `0` vào hàng đợi, chỉ quét DP một lượt, đánh dấu "đã thăm" lúc lấy ra khỏi hàng đợi thay vì lúc đưa vào.

## 2. Đề bài & ví dụ

Cho ma trận nhị phân `mat` kích thước `m x n`. Hãy trả về ma trận cùng kích thước, trong đó ô `(i, j)` chứa **khoảng cách từ ô đó tới số `0` gần nhất**. Khoảng cách giữa hai ô chung cạnh (kề trên/dưới/trái/phải) bằng `1`.

**Ví dụ 1**

- Input: `mat = [[0,0,0],[0,1,0],[0,0,0]]`
- Output: `[[0,0,0],[0,1,0],[0,0,0]]`
- Giải thích: ô giữa (số `1`) kề ngay số `0` nên khoảng cách là `1`.

**Ví dụ 2**

- Input: `mat = [[0,0,0],[0,1,0],[1,1,1]]`
- Output: `[[0,0,0],[0,1,0],[1,2,1]]`
- Giải thích: ô `(2,1)` cách số `0` gần nhất (như ô `(1,0)` hay `(1,2)`) hai bước nên là `2`; hai ô `(2,0)`, `(2,2)` kề số `0` nên là `1`.

**Ràng buộc:**

- `m == mat.length`, `n == mat[i].length`
- `1 <= m, n <= 10^4`
- `1 <= m * n <= 10^4`
- `mat[i][j]` là `0` hoặc `1`
- Có **ít nhất một** số `0` trong `mat`

Chú ý: tích `m * n <= 10^4` nên có thể là lưới `100 x 100`, hoặc `1 x 10000`, hoặc `10000 x 1`. Một lời giải `O((mn)^2)` có thể cần tới `10^8` thao tác — quá chậm với Python.

In [1]:
from typing import List
from collections import deque
from itertools import pairwise
import random

def show(grid, title=None):
    if title:
        print(title)
    for row in grid:
        print(" ".join(f"{x:2d}" for x in row))

examples = [
    ([[0, 0, 0], [0, 1, 0], [0, 0, 0]], [[0, 0, 0], [0, 1, 0], [0, 0, 0]]),
    ([[0, 0, 0], [0, 1, 0], [1, 1, 1]], [[0, 0, 0], [0, 1, 0], [1, 2, 1]]),
]
for mat, expected in examples:
    show(mat, "mat =")
    show(expected, "kỳ vọng =")
    print()

mat =
 0  0  0
 0  1  0
 0  0  0
kỳ vọng =
 0  0  0
 0  1  0
 0  0  0

mat =
 0  0  0
 0  1  0
 1  1  1
kỳ vọng =
 0  0  0
 0  1  0
 1  2  1



## 3. Kiến thức nền

### 3.1. Ma trận là list lồng list

`mat[i][j]` là phần tử ở **hàng `i`**, **cột `j`** (đếm từ `0`). Số hàng `m = len(mat)`, số cột `n = len(mat[0])`. Duyệt toàn bộ ô bằng hai vòng `for`, hoặc dùng `enumerate` để lấy cả chỉ số lẫn giá trị.

In [2]:
mat = [[0, 0, 0], [0, 1, 0], [1, 1, 1]]
m, n = len(mat), len(mat[0])
print("m =", m, "| n =", n, "| mat[2][1] =", mat[2][1])
for i, row in enumerate(mat):
    for j, x in enumerate(row):
        if x == 0:
            print(f"có số 0 ở ({i},{j})")

m = 3 | n = 3 | mat[2][1] = 1
có số 0 ở (0,0)
có số 0 ở (0,1)
có số 0 ở (0,2)
có số 0 ở (1,0)
có số 0 ở (1,2)


### 3.2. Tạo ma trận mới: cái bẫy `[[-1] * n] * m`

Ta cần một ma trận `ans` cùng kích thước, ban đầu toàn `-1` ("chưa biết"). Cách **sai** phổ biến: `[[-1] * n] * m`. Phép nhân `* m` **sao chép tham chiếu** tới cùng một list hàng, nên `m` hàng thực chất là **cùng một đối tượng**: sửa một ô là cả cột bị sửa theo. Cách đúng: `[[-1] * n for _ in range(m)]` tạo `m` list độc lập.

In [3]:
bad = [[-1] * 3] * 2
bad[0][1] = 7
print("sai :", bad, "| hai hàng cùng một đối tượng?", bad[0] is bad[1])

good = [[-1] * 3 for _ in range(2)]
good[0][1] = 7
print("đúng:", good, "| hai hàng cùng một đối tượng?", good[0] is good[1])

sai : [[-1, 7, -1], [-1, 7, -1]] | hai hàng cùng một đối tượng? True
đúng: [[-1, 7, -1], [-1, -1, -1]] | hai hàng cùng một đối tượng? False


### 3.3. Bốn hướng đi và kiểm tra biên

Từ ô `(i, j)`, bốn ô kề là `(i-1, j)` (trên), `(i+1, j)` (dưới), `(i, j-1)` (trái), `(i, j+1)` (phải). Ta lưu bốn **độ dời** `(dx, dy)` trong một tuple rồi lặp. Trước khi dùng ô kề `(x, y)` phải kiểm tra **trong biên**: `0 <= x < m and 0 <= y < n` (Python cho viết so sánh dây chuyền).

Mẹo trong lời giải mẫu của đề: `dirs = (-1, 0, 1, 0, -1)` rồi `pairwise(dirs)` sinh các cặp liên tiếp `(-1,0), (0,1), (1,0), (0,-1)`. Bản nộp của ta dùng tuple tường minh cho dễ đọc.

In [4]:
dirs = (-1, 0, 1, 0, -1)
print(list(pairwise(dirs)))                       # [(-1, 0), (0, 1), (1, 0), (0, -1)]

DIRS = ((-1, 0), (0, 1), (1, 0), (0, -1))          # trên, phải, dưới, trái
m, n = 3, 3
i, j = 0, 0                                        # góc trên trái
for dx, dy in DIRS:
    x, y = i + dx, j + dy
    print((x, y), "trong biên" if 0 <= x < m and 0 <= y < n else "ngoài biên")

[(-1, 0), (0, 1), (1, 0), (0, -1)]
(-1, 0) ngoài biên
(0, 1) trong biên
(1, 0) trong biên
(0, -1) ngoài biên


### 3.4. `deque` — hàng đợi cho BFS

`collections.deque` là hàng đợi hai đầu: `append(x)` thêm vào cuối, `popleft()` lấy ra từ đầu, cả hai `O(1)`. (Dùng `list.pop(0)` thì `O(n)` vì phải dời cả list.) BFS luôn xử lý phần tử **vào trước, ra trước** (FIFO).

In [5]:
q = deque()
q.append((0, 0)); q.append((0, 1)); q.append((1, 0))
print(q)
print(q.popleft(), "->", q)
print("rỗng?", not q, "| độ dài:", len(q))

deque([(0, 0), (0, 1), (1, 0)])
(0, 0) -> deque([(0, 1), (1, 0)])
rỗng? False | độ dài: 2


### 3.5. BFS tìm đường ngắn nhất khi mọi bước có giá `1`

BFS (Breadth-First Search, duyệt theo chiều rộng) từ một nguồn: xử lý các ô cách nguồn `0` bước, rồi tất cả ô cách `1` bước, rồi `2` bước, ... **Lần đầu tiên** BFS chạm một ô chính là đường ngắn nhất tới ô đó (vì ta đã duyệt hết mọi ô gần hơn từ trước). Ví dụ đơn giản trên một hàng `[·,·,·,·,·]` bắt đầu từ ô `0`.

In [6]:
# BFS một nguồn trên hàng ngang 6 ô, nguồn ở ô 2
n = 6
dist = [-1] * n
dist[2] = 0
q = deque([2])
while q:
    u = q.popleft()
    for v in (u - 1, u + 1):
        if 0 <= v < n and dist[v] == -1:
            dist[v] = dist[u] + 1
            q.append(v)
print(dist)       # [2, 1, 0, 1, 2, 3]

[2, 1, 0, 1, 2, 3]


### 3.6. Khoảng cách Manhattan

Trên lưới **không có vật cản**, đi từ `(a, b)` tới `(c, d)` bằng các bước ngang/dọc cần **ít nhất** `|a - c| + |b - d|` bước (gọi là **khoảng cách Manhattan**) và luôn đạt được (đi dọc rồi đi ngang). Vì bài này không có vật cản nên khoảng cách đường đi ngắn nhất **bằng đúng** khoảng cách Manhattan. Ta sẽ dùng công thức này làm "oracle" độc lập để kiểm tra các lời giải khác.

In [7]:
def manhattan(a, b, c, d):
    return abs(a - c) + abs(b - d)

print(manhattan(2, 1, 1, 0))   # (2,1) tới (1,0): 1 + 1 = 2
print(manhattan(0, 0, 3, 4))   # 7

2
7


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 2:

```
0 0 0        0 0 0
0 1 0   ->   0 1 0
1 1 1        1 2 1
```

- Số `0` → khoảng cách `0`.
- Ô `(1,1)`: kề `(0,1)` là `0` → `1`.
- Ô `(2,0)`: kề `(1,0)` là `0` → `1`. Ô `(2,2)`: kề `(1,2)` là `0` → `1`.
- Ô `(2,1)`: các ô kề là `(1,1)` (khoảng cách 1), `(2,0)` (1), `(2,2)` (1) — không kề `0` nào → `1 + 1 = 2`.

Quy luật: **một ô có khoảng cách `d` khi và chỉ khi nó kề một ô có khoảng cách `d - 1` và không kề ô nào có khoảng cách nhỏ hơn**. Ta có thể "bóc lớp": lớp `0` là các số `0`; lớp `1` là các ô `1` kề lớp `0`; lớp `2` là các ô chưa gán kề lớp `1`; ... Hãy xem sóng lan ra trên một lưới có hai số `0` ở hai góc.

In [8]:
def peel_demo(mat):
    """Bóc lớp: ở lượt d, gán d cho các ô chưa biết kề ô có giá trị d - 1."""
    m, n = len(mat), len(mat[0])
    ans = [[0 if x == 0 else -1 for x in row] for row in mat]
    show(ans, "lớp 0 (các số 0):")
    d = 0
    while any(-1 in row for row in ans):
        d += 1
        cur = []
        for i in range(m):
            for j in range(n):
                if ans[i][j] == -1 and any(
                        0 <= i + a < m and 0 <= j + b < n and ans[i + a][j + b] == d - 1
                        for a, b in DIRS):
                    cur.append((i, j))
        for i, j in cur:
            ans[i][j] = d
        print()
        show(ans, f"sau lớp {d}:")
    return ans

res = peel_demo([[0, 1, 1, 1], [1, 1, 1, 1], [1, 1, 1, 0]])

lớp 0 (các số 0):
 0 -1 -1 -1
-1 -1 -1 -1
-1 -1 -1  0

sau lớp 1:
 0  1 -1 -1
 1 -1 -1  1
-1 -1  1  0

sau lớp 2:
 0  1  2  2
 1  2  2  1
 2  2  1  0


Cách "bóc lớp" cho đáp án đúng nhưng **mỗi lượt quét lại toàn bộ lưới**: nếu khoảng cách lớn nhất là `D` thì tốn `O(m * n * D)`, với lưới `1 x 10000` có một số `0` ở đầu thì `D ≈ 10^4`, tức `10^8`. Ta cần cách chỉ "đụng" vào các ô **ở mép sóng**, mà đó chính là việc của BFS.

## 5. Cách 1 — Brute Force: BFS riêng cho từng ô `1`

### Ý tưởng

Với mỗi ô `(i, j)` có giá trị `1`, chạy một BFS **bắt đầu từ ô đó** cho tới khi gặp một ô `0` đầu tiên; số bước chính là đáp án của ô. Ô `0` thì đáp án `0`. Đây là phản xạ tự nhiên: "tìm đường ngắn nhất từ ô này tới `0` gần nhất" → BFS.

### Code

In [9]:
def bfs_each(mat):
    m, n = len(mat), len(mat[0])
    ans = [[0] * n for _ in range(m)]
    for si in range(m):
        for sj in range(n):
            if mat[si][sj] == 0:
                continue
            seen = [[False] * n for _ in range(m)]       # mỗi nguồn cần mảng "đã thăm" riêng
            seen[si][sj] = True
            q = deque([(si, sj, 0)])
            while q:
                i, j, d = q.popleft()
                if mat[i][j] == 0:
                    ans[si][sj] = d
                    break
                for dx, dy in DIRS:
                    x, y = i + dx, j + dy
                    if 0 <= x < m and 0 <= y < n and not seen[x][y]:
                        seen[x][y] = True
                        q.append((x, y, d + 1))
    return ans

for mat, expected in examples:
    got = bfs_each(mat)
    show(got, "kết quả:")
    assert got == expected

kết quả:
 0  0  0
 0  1  0
 0  0  0
kết quả:
 0  0  0
 0  1  0
 1  2  1


### Độ phức tạp và nút thắt

- Mỗi BFS có thể thăm tới `O(mn)` ô, và có tới `O(mn)` ô `1` → Time `O((mn)^2)`, Space `O(mn)` (mảng `seen` mỗi lần).
- Với `mn = 10^4`: tới `10^8` thao tác BFS — quá chậm với Python (Time Limit Exceeded).

**Nút thắt:** các BFS **lặp lại công việc**. Các ô `1` ở gần nhau chia sẻ gần hết đường đi tới số `0` nhưng mỗi BFS tìm lại từ đầu. Hãy đếm cụ thể để thấy mức lãng phí.

In [10]:
def bfs_each_cost(mat):
    """Đếm tổng số ô bị lấy ra khỏi hàng đợi qua mọi BFS riêng lẻ."""
    m, n = len(mat), len(mat[0])
    pops = 0
    for si in range(m):
        for sj in range(n):
            if mat[si][sj] == 0:
                continue
            seen = {(si, sj)}
            q = deque([(si, sj, 0)])
            while q:
                i, j, d = q.popleft()
                pops += 1
                if mat[i][j] == 0:
                    break
                for dx, dy in DIRS:
                    x, y = i + dx, j + dy
                    if 0 <= x < m and 0 <= y < n and (x, y) not in seen:
                        seen.add((x, y))
                        q.append((x, y, d + 1))
    return pops

for side in (5, 10, 20, 30):
    g = [[1] * side for _ in range(side)]
    g[0][0] = 0                                   # chỉ một số 0 ở góc
    print(f"{side}x{side} ({side*side} ô): tổng số lần lấy khỏi hàng đợi = {bfs_each_cost(g)}")

5x5 (25 ô): tổng số lần lấy khỏi hàng đợi = 400
10x10 (100 ô): tổng số lần lấy khỏi hàng đợi = 6849
20x20 (400 ô): tổng số lần lấy khỏi hàng đợi = 114399


30x30 (900 ô): tổng số lần lấy khỏi hàng đợi = 588149


Chỉ tăng cạnh lưới gấp đôi mà chi phí tăng gần **16 lần** (xấp xỉ `(mn)^2`). Với `100 x 100` nó vượt `10^7` thao tác chỉ để đếm, thực tế còn tệ hơn. Cần ý tưởng khác.

## 6. Ý tưởng thuật toán / bước đột phá

**Đảo chiều tìm kiếm.** Thay vì mỗi ô `1` đi tìm số `0`, hãy cho **mọi số `0` cùng xuất phát một lúc**:

1. Tất cả số `0` có khoảng cách `0`, bỏ hết vào hàng đợi (gọi là **BFS đa nguồn**, *multi-source BFS*).
2. Lấy ô ra khỏi hàng đợi, xem 4 ô kề; ô kề **chưa có đáp án** thì gán `đáp_án_của_ô_hiện_tại + 1` rồi bỏ vào hàng đợi.
3. Lặp tới khi hàng đợi rỗng.

Vì sao ổn? Coi như có một "siêu nguồn" nối tới mọi số `0` bằng cạnh độ dài `0`. BFS từ siêu nguồn này cho khoảng cách ngắn nhất tới từng ô — và đó chính là khoảng cách tới số `0` **gần nhất**. Mỗi ô chỉ vào hàng đợi **đúng một lần**, nên tổng công việc chỉ `O(mn)` thay vì `O((mn)^2)`.

Cụ thể về trạng thái: ta dùng `ans[i][j] == -1` làm dấu "chưa thăm"; ô nào đã có giá trị khác `-1` thì đã được đưa vào hàng đợi (hoặc là số `0`).

In [11]:
# Sóng lan theo từng lớp: hàng đợi chứa lần lượt các ô khoảng cách 0, rồi 1, rồi 2...
def wave_by_layer(mat):
    m, n = len(mat), len(mat[0])
    ans = [[-1] * n for _ in range(m)]
    layer = []
    for i in range(m):
        for j in range(n):
            if mat[i][j] == 0:
                ans[i][j] = 0
                layer.append((i, j))
    d = 0
    while layer:
        print(f"lớp {d}: {layer}")
        nxt = []
        for i, j in layer:
            for dx, dy in DIRS:
                x, y = i + dx, j + dy
                if 0 <= x < m and 0 <= y < n and ans[x][y] == -1:
                    ans[x][y] = d + 1
                    nxt.append((x, y))
        layer, d = nxt, d + 1
    return ans

show(wave_by_layer([[0, 1, 1, 1], [1, 1, 1, 1], [1, 1, 1, 0]]), "kết quả:")

lớp 0: [(0, 0), (2, 3)]
lớp 1: [(0, 1), (1, 0), (1, 3), (2, 2)]
lớp 2: [(0, 2), (1, 1), (2, 0), (0, 3), (1, 2), (2, 1)]
kết quả:
 0  1  2  2
 1  2  2  1
 2  2  1  0


## 7. Các cách tiếp theo

Phần này có thêm 4 cách (tổng 5 cách): **BFS đa nguồn** (lời giải nộp), **DP hai lượt quét**, **tách theo cột rồi hàng**, và **công thức Manhattan vét cạn** (oracle).

### 7.1. Cách 2 — BFS đa nguồn (lời giải nộp)

**Giải thích từng đoạn.**

1. `ans = [[-1] * n for _ in range(m)]`: ma trận kết quả, `-1` = chưa thăm.
2. Quét `mat`: mọi ô `0` gán `ans = 0` và bỏ vào hàng đợi `q`.
3. `while q`: lấy `(i, j)`; với mỗi hướng `(x, y)` trong biên và `ans[x][y] == -1` thì gán `ans[x][y] = ans[i][j] + 1` và `append`.
4. Trả `ans`.

Mấu chốt: **đánh dấu đã thăm ngay lúc đưa vào hàng đợi** (gán `ans`), để mỗi ô vào hàng đợi đúng một lần.

In [12]:
def multi_bfs(mat):
    m, n = len(mat), len(mat[0])
    ans = [[-1] * n for _ in range(m)]
    q = deque()
    for i in range(m):
        for j in range(n):
            if mat[i][j] == 0:
                ans[i][j] = 0
                q.append((i, j))
    while q:
        i, j = q.popleft()
        for dx, dy in DIRS:
            x, y = i + dx, j + dy
            if 0 <= x < m and 0 <= y < n and ans[x][y] == -1:
                ans[x][y] = ans[i][j] + 1
                q.append((x, y))
    return ans

for mat, expected in examples:
    got = multi_bfs(mat)
    show(got, "kết quả:")
    assert got == expected

kết quả:
 0  0  0
 0  1  0
 0  0  0
kết quả:
 0  0  0
 0  1  0
 1  2  1


In [13]:
class Solution:
    def updateMatrix(self, mat: List[List[int]]) -> List[List[int]]:
        m, n = len(mat), len(mat[0])
        ans = [[-1] * n for _ in range(m)]
        q = deque()
        for i in range(m):
            for j in range(n):
                if mat[i][j] == 0:
                    ans[i][j] = 0
                    q.append((i, j))
        while q:
            i, j = q.popleft()
            for dx, dy in ((-1, 0), (0, 1), (1, 0), (0, -1)):
                x, y = i + dx, j + dy
                if 0 <= x < m and 0 <= y < n and ans[x][y] == -1:
                    ans[x][y] = ans[i][j] + 1
                    q.append((x, y))
        return ans

print(Solution().updateMatrix([[0, 0, 0], [0, 1, 0], [1, 1, 1]]))

[[0, 0, 0], [0, 1, 0], [1, 2, 1]]


**Độ phức tạp:** Time `O(mn)` (mỗi ô vào/ra hàng đợi một lần, mỗi lần xét 4 hướng), Space `O(mn)` (ma trận `ans` và hàng đợi).

### 7.2. Cách 3 — Quy hoạch động (DP) hai lượt quét

**Ý tưởng.** Đường ngắn nhất tới số `0` gần nhất đi tới một trong 4 "góc phần tư": số `0` nằm ở **trên-trái**, **trên-phải**, **dưới-trái** hay **dưới-phải** của ô. Ta quét hai lượt:

- **Lượt 1** (trên→dưới, trái→phải): `dist[i][j] = min(dist[i][j], dist[i-1][j] + 1, dist[i][j-1] + 1)`. Lượt này lấy thông tin từ ô **trên** và **trái**.
- **Lượt 2** (dưới→trên, phải→trái): `dist[i][j] = min(dist[i][j], dist[i+1][j] + 1, dist[i][j+1] + 1)`. Lượt này lấy thông tin từ ô **dưới** và **phải**, kể cả những gì lượt 1 đã tích lũy.

Khởi tạo: ô `0` → `0`; ô `1` → `INF` (một số lớn hơn mọi khoảng cách có thể, ví dụ `m + n`). Hai lượt đủ vì (xem chi tiết ở mục 10): số `0` ở trên-trái được lượt 1 xử lý; số `0` ở dưới-phải do lượt 2; số `0` ở trên-phải hoặc dưới-trái được tạo bởi **kết hợp** hai lượt (đi dọc bằng lượt 1 rồi đi ngang bằng lượt 2, hoặc ngược lại).

In [14]:
def dp_two_pass(mat):
    m, n = len(mat), len(mat[0])
    INF = m + n                                      # lớn hơn mọi khoảng cách thật (tối đa m + n - 2)
    dist = [[0 if mat[i][j] == 0 else INF for j in range(n)] for i in range(m)]
    for i in range(m):                               # lượt 1: trên -> dưới, trái -> phải
        for j in range(n):
            if i > 0:
                dist[i][j] = min(dist[i][j], dist[i - 1][j] + 1)
            if j > 0:
                dist[i][j] = min(dist[i][j], dist[i][j - 1] + 1)
    for i in range(m - 1, -1, -1):                   # lượt 2: dưới -> trên, phải -> trái
        for j in range(n - 1, -1, -1):
            if i < m - 1:
                dist[i][j] = min(dist[i][j], dist[i + 1][j] + 1)
            if j < n - 1:
                dist[i][j] = min(dist[i][j], dist[i][j + 1] + 1)
    return dist

for mat, expected in examples:
    got = dp_two_pass(mat)
    show(got, "kết quả:")
    assert got == expected

kết quả:
 0  0  0
 0  1  0
 0  0  0
kết quả:
 0  0  0
 0  1  0
 1  2  1


In [15]:
class Solution:
    def updateMatrix(self, mat: List[List[int]]) -> List[List[int]]:
        m, n = len(mat), len(mat[0])
        INF = m + n
        dist = [[0 if mat[i][j] == 0 else INF for j in range(n)] for i in range(m)]
        for i in range(m):
            for j in range(n):
                if i > 0:
                    dist[i][j] = min(dist[i][j], dist[i - 1][j] + 1)
                if j > 0:
                    dist[i][j] = min(dist[i][j], dist[i][j - 1] + 1)
        for i in range(m - 1, -1, -1):
            for j in range(n - 1, -1, -1):
                if i < m - 1:
                    dist[i][j] = min(dist[i][j], dist[i + 1][j] + 1)
                if j < n - 1:
                    dist[i][j] = min(dist[i][j], dist[i][j + 1] + 1)
        return dist

print(Solution().updateMatrix([[0, 0, 0], [0, 1, 0], [1, 1, 1]]))

[[0, 0, 0], [0, 1, 0], [1, 2, 1]]


**Độ phức tạp:** Time `O(mn)` (hai lượt quét), Space `O(1)` thêm nếu sửa ngay trên ma trận kết quả (ta dùng `dist` làm đầu ra nên không tính thêm). Không dùng hàng đợi nên nhanh và ít bộ nhớ hơn BFS. Lưu ý: cách này **chỉ đúng khi lưới không có vật cản** (bài này thỏa).

### 7.3. Cách 4 — Tách theo cột rồi theo hàng (biến đổi khoảng cách có thể phân tách)

**Ý tưởng.** Khoảng cách Manhattan từ `(i, j)` tới số `0` tại `(a, b)` là `|i - a| + |j - b|`. Ta chia thành hai bước độc lập:

1. **Theo cột:** `col[i][j]` = khoảng cách **dọc** từ `(i, j)` tới số `0` gần nhất **trong cùng cột `j`** (`INF` nếu cột không có `0`). Tính bằng hai lượt quét dọc (xuống rồi lên).
2. **Theo hàng:** `ans[i][j] = min` trên mọi cột `b` của `|j - b| + col[i][b]`. Với mỗi hàng ta tính bằng hai lượt quét ngang (trái→phải rồi phải→trái).

Vì `min` trên mọi số `0` `(a, b)` của `|i - a| + |j - b|` bằng `min` trên mọi cột `b` của `|j - b| + (min trên a của |i - a|)` mà `min trên a` chính là `col[i][b]`. Hai bước này không hề dùng BFS hay DP "4 phía" — chỉ là biến đổi 1 chiều lặp lại hai lần.

In [16]:
def separable(mat):
    m, n = len(mat), len(mat[0])
    INF = m + n
    col = [[0 if mat[i][j] == 0 else INF for j in range(n)] for i in range(m)]
    # Bước 1: khoảng cách dọc trong từng cột
    for i in range(1, m):
        for j in range(n):
            col[i][j] = min(col[i][j], col[i - 1][j] + 1)
    for i in range(m - 2, -1, -1):
        for j in range(n):
            col[i][j] = min(col[i][j], col[i + 1][j] + 1)
    # Bước 2: lan theo chiều ngang trong từng hàng
    for i in range(m):
        row = col[i]
        for j in range(1, n):
            row[j] = min(row[j], row[j - 1] + 1)
        for j in range(n - 2, -1, -1):
            row[j] = min(row[j], row[j + 1] + 1)
    return col

for mat, expected in examples:
    got = separable(mat)
    show(got, "kết quả:")
    assert got == expected

# Quan sát bước trung gian: chỉ sau khi tính theo cột
demo = [[0, 1, 1], [1, 1, 1], [1, 1, 0]]
m, n = 3, 3
colonly = [[0 if demo[i][j] == 0 else 6 for j in range(n)] for i in range(m)]
for i in range(1, m):
    for j in range(n):
        colonly[i][j] = min(colonly[i][j], colonly[i - 1][j] + 1)
for i in range(m - 2, -1, -1):
    for j in range(n):
        colonly[i][j] = min(colonly[i][j], colonly[i + 1][j] + 1)
show(colonly, "chỉ theo cột (6 = INF, cột giữa không có số 0):")
show(separable(demo), "sau khi lan theo hàng:")

kết quả:
 0  0  0
 0  1  0
 0  0  0
kết quả:
 0  0  0
 0  1  0
 1  2  1
chỉ theo cột (6 = INF, cột giữa không có số 0):
 0  6  2
 1  6  1
 2  6  0
sau khi lan theo hàng:
 0  1  2
 1  2  1
 2  1  0


In [17]:
class Solution:
    def updateMatrix(self, mat: List[List[int]]) -> List[List[int]]:
        m, n = len(mat), len(mat[0])
        INF = m + n
        col = [[0 if mat[i][j] == 0 else INF for j in range(n)] for i in range(m)]
        for i in range(1, m):
            for j in range(n):
                col[i][j] = min(col[i][j], col[i - 1][j] + 1)
        for i in range(m - 2, -1, -1):
            for j in range(n):
                col[i][j] = min(col[i][j], col[i + 1][j] + 1)
        for i in range(m):
            row = col[i]
            for j in range(1, n):
                row[j] = min(row[j], row[j - 1] + 1)
            for j in range(n - 2, -1, -1):
                row[j] = min(row[j], row[j + 1] + 1)
        return col

print(Solution().updateMatrix([[0, 0, 0], [0, 1, 0], [1, 1, 1]]))

[[0, 0, 0], [0, 1, 0], [1, 2, 1]]


**Độ phức tạp:** Time `O(mn)` (bốn lượt quét một chiều), Space `O(mn)` cho `col` làm đầu ra (không cần hàng đợi). Hay ở chỗ nó cho thấy bài toán thực chất là **biến đổi khoảng cách (distance transform)**; kỹ thuật này dùng nhiều trong xử lý ảnh.

### 7.4. Cách 5 — Công thức Manhattan vét cạn (oracle)

**Ý tưởng.** Vì lưới không có vật cản, khoảng cách tới số `0` `(a, b)` là `|i - a| + |j - b|`. Gom danh sách tọa độ mọi số `0`, rồi với **mỗi ô** lấy `min` trên toàn bộ danh sách. Cách này **hoàn toàn độc lập** với BFS và DP nên rất hợp để kiểm tra chéo; nhưng Time `O(mn * z)` (với `z` là số ô `0`), tệ nhất khoảng `O((mn)^2)`.

In [18]:
def manhattan_brute(mat):
    m, n = len(mat), len(mat[0])
    zeros = [(i, j) for i in range(m) for j in range(n) if mat[i][j] == 0]
    return [[min(abs(i - a) + abs(j - b) for a, b in zeros) for j in range(n)] for i in range(m)]

for mat, expected in examples:
    got = manhattan_brute(mat)
    show(got, "kết quả:")
    assert got == expected

kết quả:
 0  0  0
 0  1  0
 0  0  0
kết quả:
 0  0  0
 0  1  0
 1  2  1


In [19]:
class Solution:
    def updateMatrix(self, mat: List[List[int]]) -> List[List[int]]:
        m, n = len(mat), len(mat[0])
        zeros = [(i, j) for i in range(m) for j in range(n) if mat[i][j] == 0]
        return [[min(abs(i - a) + abs(j - b) for a, b in zeros) for j in range(n)] for i in range(m)]

print(Solution().updateMatrix([[0, 0, 0], [0, 1, 0], [1, 1, 1]]))

[[0, 0, 0], [0, 1, 0], [1, 2, 1]]


**Độ phức tạp:** Time `O(mn * z) <= O((mn)^2)`, Space `O(z + mn)`. Chấp nhận được khi số `0` rất ít hoặc lưới nhỏ; không an toàn cho giới hạn lớn nhất nên **không dùng để nộp**.

## 8. TRACE từng bước

### 8.1. Trace BFS đa nguồn

Ví dụ 2: `mat = [[0,0,0],[0,1,0],[1,1,1]]`. Hướng duyệt theo thứ tự: trên, phải, dưới, trái. Hàng đợi ban đầu (duyệt theo hàng): `(0,0), (0,1), (0,2), (1,0), (1,2)`, tất cả có `ans = 0`.

| lần lấy | ô lấy ra | ô mới được gán | hàng đợi sau đó |
|---|---|---|---|
| 1 | (0,0) | không | (0,1) (0,2) (1,0) (1,2) |
| 2 | (0,1) | (1,1) = 1 | (0,2) (1,0) (1,2) (1,1) |
| 3 | (0,2) | không | (1,0) (1,2) (1,1) |
| 4 | (1,0) | (2,0) = 1 | (1,2) (1,1) (2,0) |
| 5 | (1,2) | (2,2) = 1 | (1,1) (2,0) (2,2) |
| 6 | (1,1) | (2,1) = 2 | (2,0) (2,2) (2,1) |
| 7 | (2,0) | không | (2,2) (2,1) |
| 8 | (2,2) | không | (2,1) |
| 9 | (2,1) | không | rỗng |

Nhận xét: `ans` của ô lấy ra không bao giờ giảm theo thứ tự lấy (`0,0,0,0,0,1,1,1,2`). Đó là tính chất nền tảng của BFS.

In [20]:
def trace_bfs(mat):
    m, n = len(mat), len(mat[0])
    ans = [[-1] * n for _ in range(m)]
    q = deque()
    for i in range(m):
        for j in range(n):
            if mat[i][j] == 0:
                ans[i][j] = 0
                q.append((i, j))
    print("khởi tạo: queue =", list(q))
    step = 0
    while q:
        i, j = q.popleft()
        step += 1
        new = []
        for dx, dy in DIRS:
            x, y = i + dx, j + dy
            if 0 <= x < m and 0 <= y < n and ans[x][y] == -1:
                ans[x][y] = ans[i][j] + 1
                q.append((x, y))
                new.append(((x, y), ans[x][y]))
        print(f"lần {step}: lấy {(i, j)} (d={ans[i][j]}) -> ô mới {new} | queue = {list(q)}")
    return ans

res = trace_bfs([[0, 0, 0], [0, 1, 0], [1, 1, 1]])
show(res, "kết quả:")
assert res == [[0, 0, 0], [0, 1, 0], [1, 2, 1]]

khởi tạo: queue = [(0, 0), (0, 1), (0, 2), (1, 0), (1, 2)]
lần 1: lấy (0, 0) (d=0) -> ô mới [] | queue = [(0, 1), (0, 2), (1, 0), (1, 2)]
lần 2: lấy (0, 1) (d=0) -> ô mới [((1, 1), 1)] | queue = [(0, 2), (1, 0), (1, 2), (1, 1)]
lần 3: lấy (0, 2) (d=0) -> ô mới [] | queue = [(1, 0), (1, 2), (1, 1)]
lần 4: lấy (1, 0) (d=0) -> ô mới [((2, 0), 1)] | queue = [(1, 2), (1, 1), (2, 0)]
lần 5: lấy (1, 2) (d=0) -> ô mới [((2, 2), 1)] | queue = [(1, 1), (2, 0), (2, 2)]
lần 6: lấy (1, 1) (d=1) -> ô mới [((2, 1), 2)] | queue = [(2, 0), (2, 2), (2, 1)]
lần 7: lấy (2, 0) (d=1) -> ô mới [] | queue = [(2, 2), (2, 1)]
lần 8: lấy (2, 2) (d=1) -> ô mới [] | queue = [(2, 1)]
lần 9: lấy (2, 1) (d=2) -> ô mới [] | queue = []
kết quả:
 0  0  0
 0  1  0
 1  2  1


### 8.2. Trace DP hai lượt quét

Chọn `mat = [[1,1,0],[1,1,1]]` (số `0` ở góc trên-phải, để thấy rõ lượt 2 phải "sửa sai" cho lượt 1). `m + n = 5` nên `INF = 5`.

- Sau khởi tạo: `[[5,5,0],[5,5,5]]`.
- **Lượt 1** (nhìn trên/trái): hàng 0: ô `(0,0)` và `(0,1)` không thấy `0` từ trên/trái nên vẫn `5`. Hàng 1: `(1,2)` có ô trên là `0` → `1`; `(1,0)`, `(1,1)` vẫn `5`. Kết quả: `[[5,5,0],[5,5,1]]`.
- **Lượt 2** (nhìn dưới/phải, từ góc dưới phải): `(1,2) = 1`; `(1,1) = min(5, 1 + 1) = 2`; `(1,0) = 3`; `(0,2) = 0`; `(0,1) = min(5, (1,1)+1 = 3, (0,2)+1 = 1) = 1`; `(0,0) = min(5, (1,0)+1 = 4, (0,1)+1 = 2) = 2`. Kết quả cuối: `[[2,1,0],[3,2,1]]`.

In [21]:
def trace_dp(mat):
    m, n = len(mat), len(mat[0])
    INF = m + n
    dist = [[0 if mat[i][j] == 0 else INF for j in range(n)] for i in range(m)]
    show(dist, f"khởi tạo (INF = {INF}):")
    for i in range(m):
        for j in range(n):
            old = dist[i][j]
            if i > 0:
                dist[i][j] = min(dist[i][j], dist[i - 1][j] + 1)
            if j > 0:
                dist[i][j] = min(dist[i][j], dist[i][j - 1] + 1)
            if dist[i][j] != old:
                print(f"  lượt 1: ({i},{j}) {old} -> {dist[i][j]}")
    show(dist, "sau lượt 1:")
    for i in range(m - 1, -1, -1):
        for j in range(n - 1, -1, -1):
            old = dist[i][j]
            if i < m - 1:
                dist[i][j] = min(dist[i][j], dist[i + 1][j] + 1)
            if j < n - 1:
                dist[i][j] = min(dist[i][j], dist[i][j + 1] + 1)
            if dist[i][j] != old:
                print(f"  lượt 2: ({i},{j}) {old} -> {dist[i][j]}")
    show(dist, "sau lượt 2:")
    return dist

res = trace_dp([[1, 1, 0], [1, 1, 1]])
assert res == [[2, 1, 0], [3, 2, 1]]

khởi tạo (INF = 5):
 5  5  0
 5  5  5
  lượt 1: (1,2) 5 -> 1
sau lượt 1:
 5  5  0
 5  5  1
  lượt 2: (1,1) 5 -> 2
  lượt 2: (1,0) 5 -> 3
  lượt 2: (0,1) 5 -> 1
  lượt 2: (0,0) 5 -> 2
sau lượt 2:
 2  1  0
 3  2  1


## 9. Code nộp LeetCode

Lời giải nộp là **BFS đa nguồn** (Cách 2): `O(mn)` thời gian, dễ chứng minh, không phụ thuộc giả thiết "không vật cản" (nên nó cũng là nền cho các bài biến thể có tường).

In [22]:
class Solution:
    def updateMatrix(self, mat: List[List[int]]) -> List[List[int]]:
        m, n = len(mat), len(mat[0])
        ans = [[-1] * n for _ in range(m)]
        q = deque()
        for i in range(m):
            for j in range(n):
                if mat[i][j] == 0:
                    ans[i][j] = 0
                    q.append((i, j))
        while q:
            i, j = q.popleft()
            for dx, dy in ((-1, 0), (0, 1), (1, 0), (0, -1)):
                x, y = i + dx, j + dy
                if 0 <= x < m and 0 <= y < n and ans[x][y] == -1:
                    ans[x][y] = ans[i][j] + 1
                    q.append((x, y))
        return ans


sol = Solution()
for mat, expected in examples:
    got = sol.updateMatrix(mat)
    print(mat, "->", got)
    assert got == expected

[[0, 0, 0], [0, 1, 0], [0, 0, 0]] -> [[0, 0, 0], [0, 1, 0], [0, 0, 0]]
[[0, 0, 0], [0, 1, 0], [1, 1, 1]] -> [[0, 0, 0], [0, 1, 0], [1, 2, 1]]


## 10. Vì sao đúng

### 10.1. BFS đa nguồn

Gọi `D(c)` là khoảng cách thật từ ô `c` tới số `0` gần nhất. Ta chứng minh bằng quy nạp theo `d` rằng: **các ô có `D = d` được gán `ans = d` đúng lúc, và hàng đợi luôn có dạng "các ô khoảng cách `k`, rồi các ô khoảng cách `k + 1`"**.

- **Cơ sở:** ban đầu hàng đợi chứa đúng các ô `D = 0`.
- **Bước quy nạp:** giả sử mọi ô `D <= k` đã được gán đúng và đã/đang trong hàng đợi, các ô khoảng cách `k` đứng trước các ô khoảng cách `k + 1`. Khi lấy một ô `u` có `D(u) = k` ra, mọi ô kề `v` chưa gán đều thỏa `D(v) <= k + 1`; `D(v) >= k + 1` vì nếu `D(v) <= k` thì `v` đã được gán (giả thiết quy nạp). Nên `D(v) = k + 1`, và gán `ans[v] = ans[u] + 1 = k + 1` là đúng. Mọi ô `D = k + 1` đều kề một ô `D = k` (đi theo đường ngắn nhất) nên chắc chắn được gán trong lúc xử lý các ô `D = k`.
- Mỗi ô được gán **một lần** (chỉ khi còn `-1`) nên không bị ghi đè sai. Mọi ô đều tới được số `0` (lưới liên thông và có ít nhất một số `0`) nên không còn `-1` khi kết thúc.

### 10.2. DP hai lượt quét

Xét số `0` gần nhất tại `(a, b)` của ô `(i, j)`, và đường Manhattan có `|i - a| + |j - b|` bước. Có 4 trường hợp:

- `a <= i, b <= j` (trên-trái): đi theo đường từ `(a, b)` xuống/phải; mỗi ô trên đường nhận giá trị đúng từ ô trên hoặc trái ở **lượt 1**.
- `a >= i, b >= j` (dưới-phải): đối xứng, lượt 2 lo.
- `a < i, b > j` (trên-phải): ở lượt 1 ô `(i, b)` (cùng cột với số `0`, nằm dưới nó) nhận đúng `i - a` từ ô trên. Ở lượt 2, giá trị đó lan **sang trái** dọc hàng `i` qua phép `dist[i][j+1] + 1`, cho `(i - a) + (b - j)`.
- `a > i, b < j` (dưới-trái): ở lượt 1 ô `(a, j)` (cùng hàng với số `0`, nằm bên phải nó) nhận `j - b` từ ô trái. Ở lượt 2 giá trị đó lan **lên trên** dọc cột `j` qua `dist[i+1][j] + 1`, cho `(a - i) + (j - b)`.

Ngược lại, mọi giá trị DP đều ứng với một đường đi thật (các phép `+1` qua ô kề), nên không bao giờ nhỏ hơn khoảng cách thật. Vậy kết quả đúng bằng khoảng cách nhỏ nhất.

### 10.3. Kiểm chứng bằng "chứng chỉ cục bộ"

Một ma trận `ans` là **đúng** khi và chỉ khi (a) mọi ô `0` có `ans = 0`, (b) hai ô kề có `ans` chênh nhau không quá `1`, (c) mọi ô có `ans = d > 0` đều có một ô kề với `ans = d - 1`. Ta kiểm các điều kiện này bằng code, độc lập với việc so sánh với oracle.

In [23]:
def is_valid_distance_matrix(mat, ans):
    m, n = len(mat), len(mat[0])
    for i in range(m):
        for j in range(n):
            d = ans[i][j]
            if (mat[i][j] == 0) != (d == 0):       # (a): đúng các ô 0 mới có d = 0
                return False
            nbrs = [ans[i + a][j + b] for a, b in DIRS if 0 <= i + a < m and 0 <= j + b < n]
            if any(abs(d - v) > 1 for v in nbrs):  # (b) chênh <= 1
                return False
            if d > 0 and (d - 1) not in nbrs:      # (c) có ô kề nhỏ hơn đúng 1
                return False
    return True

for mat, expected in examples:
    assert is_valid_distance_matrix(mat, expected)
    assert is_valid_distance_matrix(mat, Solution().updateMatrix(mat))
bad_ans = [[0, 0, 0], [0, 1, 0], [1, 3, 1]]
print("đáp án sai có bị phát hiện:", not is_valid_distance_matrix(examples[1][0], bad_ans))

đáp án sai có bị phát hiện: True


In [24]:
# Kiểm tra thực nghiệm tính chất của BFS: thứ tự lấy khỏi hàng đợi có d không giảm
def pop_order_distances(mat):
    m, n = len(mat), len(mat[0])
    ans = [[-1] * n for _ in range(m)]
    q = deque()
    for i in range(m):
        for j in range(n):
            if mat[i][j] == 0:
                ans[i][j] = 0
                q.append((i, j))
    order = []
    while q:
        i, j = q.popleft()
        order.append(ans[i][j])
        for dx, dy in DIRS:
            x, y = i + dx, j + dy
            if 0 <= x < m and 0 <= y < n and ans[x][y] == -1:
                ans[x][y] = ans[i][j] + 1
                q.append((x, y))
    return order

rng = random.Random(7)
for _ in range(200):
    m, n = rng.randint(1, 8), rng.randint(1, 8)
    g = [[rng.randint(0, 1) for _ in range(n)] for _ in range(m)]
    g[rng.randrange(m)][rng.randrange(n)] = 0
    order = pop_order_distances(g)
    assert order == sorted(order) and len(order) == m * n
print("thứ tự lấy ra không giảm trên 200 lưới; mỗi ô vào hàng đợi đúng 1 lần")

thứ tự lấy ra không giảm trên 200 lưới; mỗi ô vào hàng đợi đúng 1 lần


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. BFS riêng từng ô 1 | `O((mn)^2)` | `O(mn)` |
| 2. BFS đa nguồn (nộp) | `O(mn)` | `O(mn)` |
| 3. DP hai lượt quét | `O(mn)` | `O(1)` thêm (ngoài đầu ra) |
| 4. Tách cột + hàng | `O(mn)` | `O(mn)` (đầu ra) |
| 5. Manhattan vét cạn | `O(mn * z)` | `O(z + mn)` |

Với BFS đa nguồn: mỗi ô vào hàng đợi đúng 1 lần (`mn` lần `append`/`popleft`), mỗi lần lấy ra xét 4 hướng → `4mn` phép kiểm tra. Ta đếm thật để thấy.

In [25]:
def count_ops(mat):
    m, n = len(mat), len(mat[0])
    ans = [[-1] * n for _ in range(m)]
    q = deque()
    pushes = 0
    for i in range(m):
        for j in range(n):
            if mat[i][j] == 0:
                ans[i][j] = 0
                q.append((i, j)); pushes += 1
    checks = 0
    while q:
        i, j = q.popleft()
        for dx, dy in DIRS:
            checks += 1
            x, y = i + dx, j + dy
            if 0 <= x < m and 0 <= y < n and ans[x][y] == -1:
                ans[x][y] = ans[i][j] + 1
                q.append((x, y)); pushes += 1
    return pushes, checks

for m, n in [(10, 10), (100, 100), (1, 10000), (10000, 1)]:
    g = [[1] * n for _ in range(m)]
    g[0][0] = 0
    pushes, checks = count_ops(g)
    print(f"{m}x{n}: mn = {m*n}, số lần đưa vào hàng đợi = {pushes}, số lần kiểm tra hướng = {checks} (= 4*mn)")
    assert pushes == m * n and checks == 4 * m * n

10x10: mn = 100, số lần đưa vào hàng đợi = 100, số lần kiểm tra hướng = 400 (= 4*mn)
100x100: mn = 10000, số lần đưa vào hàng đợi = 10000, số lần kiểm tra hướng = 40000 (= 4*mn)
1x10000: mn = 10000, số lần đưa vào hàng đợi = 10000, số lần kiểm tra hướng = 40000 (= 4*mn)
10000x1: mn = 10000, số lần đưa vào hàng đợi = 10000, số lần kiểm tra hướng = 40000 (= 4*mn)


## 12. Bộ test

Gồm: ví dụ đề, biên (`1 x 1`, một hàng, một cột, toàn `0`, một `0` duy nhất ở góc/giữa), phản ví dụ cho lời giải sai (DP chỉ một lượt), đối chiếu 5 cách bằng `assert` trên rất nhiều lưới nhỏ (kể cả **vét cạn mọi lưới nhỏ**), test ngẫu nhiên có seed đúng ràng buộc, và test kích thước lớn.

In [26]:
ALL_METHODS = {
    "bfs_each": bfs_each,
    "multi_bfs": multi_bfs,
    "dp_two_pass": dp_two_pass,
    "separable": separable,
    "manhattan": manhattan_brute,
    "leetcode": Solution().updateMatrix,
}

def run_all(mat, expected=None):
    """Chạy mọi cách trên bản sao của mat; trả về kết quả chung (assert các cách khớp nhau)."""
    results = {name: fn([row[:] for row in mat]) for name, fn in ALL_METHODS.items()}
    ref = results["manhattan"]
    for name, got in results.items():
        assert got == ref, (name, mat, got, ref)
    if expected is not None:
        assert ref == expected, (mat, ref, expected)
    assert is_valid_distance_matrix(mat, ref)
    return ref

fixed_cases = [
    (examples[0][0], examples[0][1]),
    (examples[1][0], examples[1][1]),
    ([[0]], [[0]]),                                            # 1 x 1
    ([[0, 0, 0, 0]], [[0, 0, 0, 0]]),                          # toàn 0
    ([[1, 1, 0, 1, 1]], [[2, 1, 0, 1, 2]]),                    # một hàng
    ([[1], [1], [0], [1]], [[2], [1], [0], [1]]),              # một cột
    ([[0, 1], [1, 1]], [[0, 1], [1, 2]]),                      # một số 0 ở góc
    ([[1, 1, 1], [1, 0, 1], [1, 1, 1]], [[2, 1, 2], [1, 0, 1], [2, 1, 2]]),   # một số 0 ở giữa
    ([[0, 1, 1, 1], [1, 1, 1, 1], [1, 1, 1, 0]], [[0, 1, 2, 2], [1, 2, 2, 1], [2, 2, 1, 0]]),   # hai số 0 ở hai góc
]
for mat, expected in fixed_cases:
    run_all(mat, expected)
print("test cố định:", len(fixed_cases), "bộ x", len(ALL_METHODS), "cách: OK")

test cố định: 9 bộ x 6 cách: OK


In [27]:
# Phản ví dụ: DP CHỈ MỘT LƯỢT (trên-trái) -> sai khi số 0 nằm ở dưới/phải
def dp_one_pass(mat):
    m, n = len(mat), len(mat[0])
    INF = m + n
    dist = [[0 if mat[i][j] == 0 else INF for j in range(n)] for i in range(m)]
    for i in range(m):
        for j in range(n):
            if i > 0:
                dist[i][j] = min(dist[i][j], dist[i - 1][j] + 1)
            if j > 0:
                dist[i][j] = min(dist[i][j], dist[i][j - 1] + 1)
    return dist

g = [[1, 1, 0]]
print("một lượt:", dp_one_pass(g), "| đúng:", manhattan_brute(g))
g = [[1, 1], [1, 0]]
print("một lượt:", dp_one_pass(g), "| đúng:", manhattan_brute(g))

một lượt: [[4, 4, 0]] | đúng: [[2, 1, 0]]
một lượt: [[4, 4], [4, 0]] | đúng: [[2, 1], [1, 0]]


In [28]:
# Vét cạn MỌI lưới nhị phân m x n với m, n <= 3 (và 1 x 6, 6 x 1) có ít nhất một số 0
from itertools import product

total = 0
shapes = [(m, n) for m in range(1, 4) for n in range(1, 4)] + [(1, 6), (6, 1), (2, 4), (4, 2)]
for m, n in shapes:
    for bits in product((0, 1), repeat=m * n):
        if 0 not in bits:
            continue
        g = [list(bits[i * n:(i + 1) * n]) for i in range(m)]
        run_all(g)
        total += 1
print("vét cạn:", total, "lưới nhị phân nhỏ, 6 cách đều khớp")

vét cạn: 1309 lưới nhị phân nhỏ, 6 cách đều khớp


In [29]:
# Ngẫu nhiên có seed, đúng ràng buộc (m * n <= 10^4, ít nhất một số 0)
rng = random.Random(542)

def random_grid(rng, m, n, p_zero):
    g = [[0 if rng.random() < p_zero else 1 for _ in range(n)] for _ in range(m)]
    g[rng.randrange(m)][rng.randrange(n)] = 0       # đảm bảo có ít nhất một số 0
    return g

for _ in range(400):
    m, n = rng.randint(1, 7), rng.randint(1, 7)
    run_all(random_grid(rng, m, n, rng.choice([0.05, 0.2, 0.5, 0.9])))
print("ngẫu nhiên nhỏ: OK")

# Lưới vừa: bỏ BFS-từng-ô (quá chậm) nhưng vẫn so các cách còn lại
fast = ["multi_bfs", "dp_two_pass", "separable", "manhattan", "leetcode"]
for _ in range(20):
    m, n = rng.randint(10, 30), rng.randint(10, 30)
    g = random_grid(rng, m, n, rng.choice([0.01, 0.1]))
    ref = manhattan_brute(g)
    for name in fast:
        assert ALL_METHODS[name]([row[:] for row in g]) == ref, name
print("ngẫu nhiên vừa: OK")

ngẫu nhiên nhỏ: OK


ngẫu nhiên vừa: OK


In [30]:
# Kích thước lớn: 100 x 100, 1 x 10000, 10000 x 1
# 1) một số 0 duy nhất: đáp án phải đúng công thức |i - a| + |j - b|
for (m, n, a, b) in [(100, 100, 0, 0), (100, 100, 99, 99), (100, 100, 37, 64), (1, 10000, 0, 0),
                     (1, 10000, 0, 6789), (10000, 1, 5000, 0)]:
    g = [[1] * n for _ in range(m)]
    g[a][b] = 0
    expected = [[abs(i - a) + abs(j - b) for j in range(n)] for i in range(m)]
    for name in ["multi_bfs", "dp_two_pass", "separable", "leetcode"]:
        assert ALL_METHODS[name]([row[:] for row in g]) == expected, (name, m, n, a, b)
print("một số 0 duy nhất, kích thước lớn: OK")

# 2) ngẫu nhiên thưa 100 x 100 (5% số 0), so với oracle Manhattan (chỉ ~500 số 0 nên đủ nhanh)
g = random_grid(rng, 100, 100, 0.05)
ref = manhattan_brute(g)
for name in ["multi_bfs", "dp_two_pass", "separable", "leetcode"]:
    assert ALL_METHODS[name]([row[:] for row in g]) == ref, name
assert is_valid_distance_matrix(g, ref)

# 3) toàn số 1 trừ một ô, và toàn số 0
g = [[0] * 100 for _ in range(100)]
assert Solution().updateMatrix(g) == g
print("lớn ngẫu nhiên và toàn 0: OK")

một số 0 duy nhất, kích thước lớn: OK


lớn ngẫu nhiên và toàn 0: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tạo ma trận bằng `[[-1] * n] * m`

Đã thấy ở mục 3.2: các hàng là cùng một đối tượng nên sửa một hàng là sửa tất cả. Hậu quả với BFS: kết quả sai hoàn toàn.

In [31]:
def wrong_alias(mat):
    m, n = len(mat), len(mat[0])
    ans = [[-1] * n] * m                    # SAI: các hàng dùng chung một list
    q = deque()
    for i in range(m):
        for j in range(n):
            if mat[i][j] == 0:
                ans[i][j] = 0
                q.append((i, j))
    while q:
        i, j = q.popleft()
        for dx, dy in DIRS:
            x, y = i + dx, j + dy
            if 0 <= x < m and 0 <= y < n and ans[x][y] == -1:
                ans[x][y] = ans[i][j] + 1
                q.append((x, y))
    return ans

g = [[0, 0, 0], [0, 1, 0], [1, 1, 1]]
print("sai  :", wrong_alias(g))
print("đúng :", multi_bfs(g))

sai  : [[0, 0, 0], [0, 0, 0], [0, 0, 0]]
đúng : [[0, 0, 0], [0, 1, 0], [1, 2, 1]]


### Lỗi 2 — Quên gán `0` cho các ô `0` (hoặc quên đưa chúng vào hàng đợi)

Nếu khởi tạo `ans` toàn `-1` và chỉ bỏ ô `0` vào hàng đợi mà **không** gán `ans[i][j] = 0`, ô `0` vẫn là `-1` và sẽ bị coi là "chưa thăm": một ô khác sẽ ghi đè khoảng cách `d + 1` lên chính số `0`.

In [32]:
def wrong_no_zero_assign(mat):
    m, n = len(mat), len(mat[0])
    ans = [[-1] * n for _ in range(m)]
    q = deque()
    for i in range(m):
        for j in range(n):
            if mat[i][j] == 0:
                q.append((i, j))             # SAI: thiếu ans[i][j] = 0
    while q:
        i, j = q.popleft()
        base = max(ans[i][j], 0)
        for dx, dy in DIRS:
            x, y = i + dx, j + dy
            if 0 <= x < m and 0 <= y < n and ans[x][y] == -1:
                ans[x][y] = base + 1
                q.append((x, y))
    return ans

g = [[0, 1], [1, 1]]
print("sai :", wrong_no_zero_assign(g), "(ô (0,0) lẽ ra là 0, ô (0,1),(1,0) lẽ ra là 1)")
print("đúng:", multi_bfs(g))

sai : [[2, 1], [1, 2]] (ô (0,0) lẽ ra là 0, ô (0,1),(1,0) lẽ ra là 1)
đúng: [[0, 1], [1, 2]]


### Lỗi 3 — DP chỉ một lượt quét

Một lượt trên-trái chỉ nhìn thấy số `0` nằm phía **trên hoặc trái**. Hàng `[1, 1, 0]` cho ra `[INF, INF, 0]` — đã chứng kiến ở mục 12. Cần thêm lượt dưới-phải.

### Lỗi 4 — Đánh dấu "đã thăm" lúc *lấy ra* thay vì lúc *đưa vào*

Nếu chỉ đánh dấu khi `popleft`, một ô có thể bị đưa vào hàng đợi **nhiều lần** (từ nhiều ô kề khác nhau). Kết quả vẫn có thể đúng nhưng hàng đợi phình to, tốn bộ nhớ/thời gian — trong BFS trên lưới lớn có thể gây TLE.

In [33]:
def wrong_mark_on_pop(mat):
    m, n = len(mat), len(mat[0])
    ans = [[-1] * n for _ in range(m)]
    visited = [[False] * n for _ in range(m)]
    q = deque()
    for i in range(m):
        for j in range(n):
            if mat[i][j] == 0:
                ans[i][j] = 0
                q.append((i, j))
    pushes = len(q)
    while q:
        i, j = q.popleft()
        if visited[i][j]:
            continue
        visited[i][j] = True                  # đánh dấu quá muộn
        for dx, dy in DIRS:
            x, y = i + dx, j + dy
            if 0 <= x < m and 0 <= y < n and not visited[x][y]:
                if ans[x][y] == -1:
                    ans[x][y] = ans[i][j] + 1
                q.append((x, y))
                pushes += 1
    return ans, pushes

g = [[1] * 30 for _ in range(30)]
g[0][0] = 0
res, pushes = wrong_mark_on_pop(g)
print("đưa vào hàng đợi:", pushes, "lần cho", 30 * 30, "ô (đúng phải là 900); kết quả vẫn khớp:", res == multi_bfs(g))

đưa vào hàng đợi: 1741 lần cho 900 ô (đúng phải là 900); kết quả vẫn khớp: True


### Lỗi 5 — Khởi tạo `INF` quá nhỏ trong DP

`INF` phải **lớn hơn mọi khoảng cách thật** (tối đa `m + n - 2`). Nếu đặt `INF = 1`, các ô xa sẽ bị cắt cụt thành giá trị sai nhưng trông "hợp lý".

In [34]:
def dp_wrong_inf(mat, INF):
    m, n = len(mat), len(mat[0])
    dist = [[0 if mat[i][j] == 0 else INF for j in range(n)] for i in range(m)]
    for i in range(m):
        for j in range(n):
            if i > 0: dist[i][j] = min(dist[i][j], dist[i - 1][j] + 1)
            if j > 0: dist[i][j] = min(dist[i][j], dist[i][j - 1] + 1)
    for i in range(m - 1, -1, -1):
        for j in range(n - 1, -1, -1):
            if i < m - 1: dist[i][j] = min(dist[i][j], dist[i + 1][j] + 1)
            if j < n - 1: dist[i][j] = min(dist[i][j], dist[i][j + 1] + 1)
    return dist

g = [[1, 1, 1, 1, 0]]
print("INF = 2 (sai) :", dp_wrong_inf(g, 2))
print("INF = m + n   :", dp_wrong_inf(g, 1 + 5))

INF = 2 (sai) : [[2, 2, 2, 1, 0]]
INF = m + n   : [[4, 3, 2, 1, 0]]


## 14. Mẹo & nhận dạng pattern

- **"Với mỗi ô, khoảng cách tới thứ gần nhất"** → nghĩ ngay tới **BFS đa nguồn**: đưa mọi "thứ" vào hàng đợi cùng lúc, đừng BFS riêng từng ô.
- Cùng họ: *Rotting Oranges* (994, nhiều nguồn là các quả cam thối), *Walls and Gates* (286), *As Far from Land as Possible* (1162), *Map of Highest Peak* (1765).
- **Lưới không vật cản + khoảng cách ngang/dọc** → DP hai lượt (trên-trái rồi dưới-phải) cũng đủ, nhanh và gọn. Có vật cản thì DP hai lượt **không còn đúng**, hãy dùng BFS.
- Quy tắc BFS trên lưới: gán/đánh dấu **ngay khi đưa vào** hàng đợi; dùng `deque.popleft()`, không dùng `list.pop(0)`.
- Khuôn mẫu 4 hướng: `for dx, dy in ((-1, 0), (0, 1), (1, 0), (0, -1))` rồi kiểm tra biên `0 <= x < m and 0 <= y < n`.
- Khi cần kiểm tra lời giải: viết oracle ngây thơ (Manhattan vét cạn) và so sánh trên hàng nghìn lưới nhỏ ngẫu nhiên, kèm "chứng chỉ cục bộ".

## 15. Biến thể

### 15.1. Khoảng cách tới số `1` gần nhất (đảo vai trò)

Chỉ cần đảo giá trị rồi dùng lại hàm: ô `1` thành nguồn.

### 15.2. Cho phép đi chéo (8 hướng)

Khoảng cách bây giờ là **Chebyshev**: `max(|dx|, |dy|)`. Chỉ cần thêm 4 hướng chéo vào danh sách hướng.

### 15.3. Map of Highest Peak (LeetCode 1765)

Cho `isWater`: ô `1` là nước (độ cao `0`), ô `0` là đất. Độ cao mỗi ô đất = khoảng cách tới ô nước gần nhất. Chính là bài này với các giá trị đảo.

In [35]:
DIRS8 = DIRS + ((-1, -1), (-1, 1), (1, -1), (1, 1))

def nearest_one(mat):
    """15.1: khoảng cách tới số 1 gần nhất (yêu cầu có ít nhất một số 1)."""
    flipped = [[1 - x for x in row] for row in mat]
    return multi_bfs(flipped)

def update_matrix_8dir(mat):
    """15.2: BFS đa nguồn với 8 hướng."""
    m, n = len(mat), len(mat[0])
    ans = [[-1] * n for _ in range(m)]
    q = deque()
    for i in range(m):
        for j in range(n):
            if mat[i][j] == 0:
                ans[i][j] = 0
                q.append((i, j))
    while q:
        i, j = q.popleft()
        for dx, dy in DIRS8:
            x, y = i + dx, j + dy
            if 0 <= x < m and 0 <= y < n and ans[x][y] == -1:
                ans[x][y] = ans[i][j] + 1
                q.append((x, y))
    return ans

def highest_peak(is_water):
    """15.3: LeetCode 1765 - nước (1) có độ cao 0; độ cao = khoảng cách tới nước gần nhất."""
    return multi_bfs([[0 if w else 1 for w in row] for row in is_water])

print(nearest_one([[1, 1, 0], [1, 1, 1]]))
print(update_matrix_8dir([[0, 1, 1], [1, 1, 1], [1, 1, 1]]))
assert highest_peak([[0, 1], [0, 0]]) == [[1, 0], [2, 1]]
assert highest_peak([[0, 0, 1], [1, 0, 0], [0, 0, 0]]) == [[1, 1, 0], [0, 1, 1], [1, 2, 2]]

# kiểm chứng 15.2 bằng công thức Chebyshev
rng2 = random.Random(1)
for _ in range(200):
    m, n = rng2.randint(1, 6), rng2.randint(1, 6)
    g = random_grid(rng2, m, n, 0.2)
    zs = [(i, j) for i in range(m) for j in range(n) if g[i][j] == 0]
    exp = [[min(max(abs(i - a), abs(j - b)) for a, b in zs) for j in range(n)] for i in range(m)]
    assert update_matrix_8dir(g) == exp
print("các biến thể 15.1-15.3 OK")

[[0, 0, 1], [0, 0, 0]]
[[0, 1, 2], [1, 1, 2], [2, 2, 2]]
các biến thể 15.1-15.3 OK


### 15.4. Có vật cản (Walls and Gates, LeetCode 286)

Khi có **tường** (ô không đi được), khoảng cách đường đi ngắn nhất không còn bằng Manhattan, và DP hai lượt cũng không đúng nữa. BFS đa nguồn **vẫn đúng** — chỉ cần không bước vào ô tường. Đây là Bài 5 của phần luyện tập.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS từng ô 1 | Mỗi ô tự BFS tới số 0 | `O((mn)^2)` | `O(mn)` | Chỉ để hiểu đề; TLE ở giới hạn lớn |
| 2. BFS đa nguồn | Mọi số 0 cùng lan ra như sóng | `O(mn)` | `O(mn)` | **Nộp bài**; đúng cả khi có vật cản |
| 3. DP hai lượt | Trên-trái rồi dưới-phải | `O(mn)` | `O(1)` thêm | Lưới không vật cản; nhanh, ngắn |
| 4. Tách cột + hàng | Khoảng cách dọc rồi lan ngang | `O(mn)` | `O(mn)` | Biến đổi khoảng cách tổng quát, xử lý ảnh |
| 5. Manhattan vét cạn | Min công thức trên mọi số 0 | `O(mn * z)` | `O(z + mn)` | Oracle kiểm tra, lưới nhỏ |

In [36]:
import time

rng3 = random.Random(2024)
g = random_grid(rng3, 100, 100, 0.02)
ref = manhattan_brute(g)
for name in ["multi_bfs", "dp_two_pass", "separable"]:
    got = ALL_METHODS[name]([row[:] for row in g])
    print(f"{name:12s} khớp oracle trên lưới 100x100:", got == ref)
    assert got == ref

multi_bfs    khớp oracle trên lưới 100x100: True
dp_two_pass  khớp oracle trên lưới 100x100: True
separable    khớp oracle trên lưới 100x100: True


## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy để kiểm tra:

```python
dirs = (-1, 0, 1, 0, -1)
print(list(zip(dirs, dirs[1:])))
print(multi_bfs([[1, 0, 1]]))
print(multi_bfs([[0, 1], [1, 1]]))
```

In [37]:
# Bài 1: viết dự đoán của bạn vào biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay DP hai lượt (dễ-vừa)

Với `mat = [[1,1,1],[1,0,1]]` (`m + n = 5`, `INF = 5`), hãy ghi ma trận **sau lượt 1** và **sau lượt 2**.

In [38]:
# Bài 2: điền kết quả tính tay (list lồng list)
hand_after_pass1 = None
hand_after_pass2 = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành vòng BFS đa nguồn: phần khởi tạo đã cho sẵn, bạn điền phần lan sóng.

In [39]:
def exercise_3_update_matrix(mat: List[List[int]]) -> List[List[int]]:
    m, n = len(mat), len(mat[0])
    ans = [[-1] * n for _ in range(m)]
    q = deque()
    for i in range(m):
        for j in range(n):
            if mat[i][j] == 0:
                ans[i][j] = 0
                q.append((i, j))
    # TODO 1: lặp khi hàng đợi còn phần tử
    #     TODO 2: lấy (i, j) ra từ ĐẦU hàng đợi
    #     TODO 3: với mỗi hướng (dx, dy), tính (x, y) và kiểm tra trong biên VÀ ans[x][y] == -1
    #         TODO 4: gán ans[x][y] = ans[i][j] + 1 và đưa (x, y) vào hàng đợi
    return ans


# assert exercise_3_update_matrix([[0, 0, 0], [0, 1, 0], [1, 1, 1]]) == [[0, 0, 0], [0, 1, 0], [1, 2, 1]]
# assert exercise_3_update_matrix([[0]]) == [[0]]
# assert exercise_3_update_matrix([[1, 1, 0]]) == [[2, 1, 0]]

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại cách 3, hãy tự viết `exercise_4_dp(mat)` bằng **DP hai lượt quét** (nhớ chọn `INF` đủ lớn).

In [40]:
def exercise_4_dp(mat: List[List[int]]) -> List[List[int]]:
    # TODO
    pass


# assert exercise_4_dp([[0, 0, 0], [0, 1, 0], [1, 1, 1]]) == [[0, 0, 0], [0, 1, 0], [1, 2, 1]]
# assert exercise_4_dp([[1, 1, 0]]) == [[2, 1, 0]]
# assert exercise_4_dp([[1, 1], [1, 0]]) == [[2, 1], [1, 0]]

### Bài 5 — Mini challenge: Walls and Gates (có tường) (khó)

Cho lưới `rooms` với: `-1` là **tường** (không đi qua được), `0` là **cổng**, `INF = 2147483647` là **phòng trống**. Trả về lưới mới, trong đó mỗi phòng trống chứa khoảng cách tới **cổng gần nhất** (đi 4 hướng, không qua tường); phòng không tới được cổng giữ nguyên `INF`. Tường và cổng giữ nguyên giá trị.

```python
INF = 2147483647
rooms = [[INF, -1, 0, INF],
         [INF, INF, INF, -1],
         [INF, -1, INF, -1],
         [0, -1, INF, INF]]
-> [[3, -1, 0, 1],
    [2, 2, 1, -1],
    [1, -1, 2, -1],
    [0, -1, 3, 4]]
```

In [41]:
INF_ROOM = 2147483647

def exercise_5_walls_and_gates(rooms: List[List[int]]) -> List[List[int]]:
    # TODO
    pass


# assert exercise_5_walls_and_gates([[INF_ROOM, -1, 0, INF_ROOM], [INF_ROOM, INF_ROOM, INF_ROOM, -1],
#                                    [INF_ROOM, -1, INF_ROOM, -1], [0, -1, INF_ROOM, INF_ROOM]]) == \
#        [[3, -1, 0, 1], [2, 2, 1, -1], [1, -1, 2, -1], [0, -1, 3, 4]]
# assert exercise_5_walls_and_gates([[-1]]) == [[-1]]
# assert exercise_5_walls_and_gates([[INF_ROOM, 0], [-1, INF_ROOM]]) == [[1, 0], [-1, 1]]

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `zip(dirs, dirs[1:])` ghép phần tử với phần tử kế bên.
- Mức 2: ở `[[1, 0, 1]]` chỉ có một số `0` ở giữa; hai ô `1` kề nó.
- Mức 3: dòng 1 `[(-1, 0), (0, 1), (1, 0), (0, -1)]`; dòng 2 `[[1, 0, 1]]`; dòng 3 `[[0, 1], [1, 2]]` (ô `(1,1)` cách `0` hai bước).

**Bài 2**
- Mức 1: lượt 1 chỉ nhìn ô trên và ô trái; hàng 0 không có `0` nào ở trên/trái.
- Mức 2: sau lượt 1 hàng 0 vẫn `[5,5,5]`; hàng 1: `[5,0,1]`.
- Mức 3: lượt 2 cho `[[2,1,2],[1,0,1]]`.

**Bài 3**
- Mức 1: giống Cách 2 (mục 7.1).
- Mức 2: `while q:` rồi `i, j = q.popleft()`.
- Mức 3: `for dx, dy in ((-1,0),(0,1),(1,0),(0,-1)): x, y = i+dx, j+dy; if 0 <= x < m and 0 <= y < n and ans[x][y] == -1: ans[x][y] = ans[i][j] + 1; q.append((x, y))`.

**Bài 4**
- Mức 1: `INF = m + n`; ô `0` → `0`, ô `1` → `INF`.
- Mức 2: lượt 1 `for i in range(m): for j in range(n):` lấy `min` với ô trên/trái `+ 1`.
- Mức 3: lượt 2 duyệt ngược `range(m - 1, -1, -1)`, lấy `min` với ô dưới/phải `+ 1`.

**Bài 5**
- Mức 1: đưa mọi **cổng** (giá trị `0`) vào hàng đợi — đây là BFS đa nguồn.
- Mức 2: ô được phép gán chỉ khi nó là phòng trống (`== INF_ROOM`); tường `-1` không bao giờ bằng `INF_ROOM` nên tự bị bỏ qua.
- Mức 3: `rooms[x][y] == INF_ROOM` vừa là điều kiện "chưa thăm" vừa là điều kiện "không phải tường"; nhớ trả về **lưới mới** (sao chép trước) nếu không muốn sửa đầu vào.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [42]:
dirs = (-1, 0, 1, 0, -1)
print(list(zip(dirs, dirs[1:])))
print(multi_bfs([[1, 0, 1]]))
print(multi_bfs([[0, 1], [1, 1]]))

assert list(zip(dirs, dirs[1:])) == [(-1, 0), (0, 1), (1, 0), (0, -1)]
assert multi_bfs([[1, 0, 1]]) == [[1, 0, 1]]
assert multi_bfs([[0, 1], [1, 1]]) == [[0, 1], [1, 2]]

[(-1, 0), (0, 1), (1, 0), (0, -1)]
[[1, 0, 1]]
[[0, 1], [1, 2]]


### Đáp án Bài 2

Sau lượt 1: `[[5,5,5],[5,0,1]]` (ô `(1,2)` nhận `1` từ ô trái là `0`; hàng 0 chưa thấy `0` nào ở trên/trái).
Sau lượt 2: `[[2,1,2],[1,0,1]]`. Ví dụ ô `(0,1)` lấy `min(5, (1,1)+1 = 1, (0,2)+1 = 3) = 1`.

In [43]:
res = trace_dp([[1, 1, 1], [1, 0, 1]])
assert res == [[2, 1, 2], [1, 0, 1]]
print("Bài 2 OK:", res)

khởi tạo (INF = 5):
 5  5  5
 5  0  5
  lượt 1: (1,2) 5 -> 1
sau lượt 1:
 5  5  5
 5  0  1
  lượt 2: (1,0) 5 -> 1
  lượt 2: (0,2) 5 -> 2
  lượt 2: (0,1) 5 -> 1
  lượt 2: (0,0) 5 -> 2
sau lượt 2:
 2  1  2
 1  0  1
Bài 2 OK: [[2, 1, 2], [1, 0, 1]]


### Đáp án Bài 3

In [44]:
def exercise_3_solution(mat: List[List[int]]) -> List[List[int]]:
    m, n = len(mat), len(mat[0])
    ans = [[-1] * n for _ in range(m)]
    q = deque()
    for i in range(m):
        for j in range(n):
            if mat[i][j] == 0:
                ans[i][j] = 0
                q.append((i, j))
    while q:
        i, j = q.popleft()
        for dx, dy in ((-1, 0), (0, 1), (1, 0), (0, -1)):
            x, y = i + dx, j + dy
            if 0 <= x < m and 0 <= y < n and ans[x][y] == -1:
                ans[x][y] = ans[i][j] + 1
                q.append((x, y))
    return ans

assert exercise_3_solution([[0, 0, 0], [0, 1, 0], [1, 1, 1]]) == [[0, 0, 0], [0, 1, 0], [1, 2, 1]]
assert exercise_3_solution([[0]]) == [[0]]
assert exercise_3_solution([[1, 1, 0]]) == [[2, 1, 0]]
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [45]:
def exercise_4_solution(mat: List[List[int]]) -> List[List[int]]:
    m, n = len(mat), len(mat[0])
    INF = m + n
    dist = [[0 if mat[i][j] == 0 else INF for j in range(n)] for i in range(m)]
    for i in range(m):
        for j in range(n):
            if i > 0:
                dist[i][j] = min(dist[i][j], dist[i - 1][j] + 1)
            if j > 0:
                dist[i][j] = min(dist[i][j], dist[i][j - 1] + 1)
    for i in range(m - 1, -1, -1):
        for j in range(n - 1, -1, -1):
            if i < m - 1:
                dist[i][j] = min(dist[i][j], dist[i + 1][j] + 1)
            if j < n - 1:
                dist[i][j] = min(dist[i][j], dist[i][j + 1] + 1)
    return dist

assert exercise_4_solution([[0, 0, 0], [0, 1, 0], [1, 1, 1]]) == [[0, 0, 0], [0, 1, 0], [1, 2, 1]]
assert exercise_4_solution([[1, 1, 0]]) == [[2, 1, 0]]
assert exercise_4_solution([[1, 1], [1, 0]]) == [[2, 1], [1, 0]]
rng4 = random.Random(44)
for _ in range(200):
    g = random_grid(rng4, rng4.randint(1, 6), rng4.randint(1, 6), 0.3)
    assert exercise_4_solution(g) == manhattan_brute(g)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [46]:
def exercise_5_solution(rooms: List[List[int]]) -> List[List[int]]:
    m, n = len(rooms), len(rooms[0])
    out = [row[:] for row in rooms]
    q = deque((i, j) for i in range(m) for j in range(n) if out[i][j] == 0)
    while q:
        i, j = q.popleft()
        for dx, dy in ((-1, 0), (0, 1), (1, 0), (0, -1)):
            x, y = i + dx, j + dy
            if 0 <= x < m and 0 <= y < n and out[x][y] == INF_ROOM:
                out[x][y] = out[i][j] + 1
                q.append((x, y))
    return out

sample = [[INF_ROOM, -1, 0, INF_ROOM], [INF_ROOM, INF_ROOM, INF_ROOM, -1],
          [INF_ROOM, -1, INF_ROOM, -1], [0, -1, INF_ROOM, INF_ROOM]]
assert exercise_5_solution(sample) == [[3, -1, 0, 1], [2, 2, 1, -1], [1, -1, 2, -1], [0, -1, 3, 4]]
assert exercise_5_solution([[-1]]) == [[-1]]
assert exercise_5_solution([[INF_ROOM, 0], [-1, INF_ROOM]]) == [[1, 0], [-1, 1]]
assert exercise_5_solution([[INF_ROOM, -1, 0]]) == [[INF_ROOM, -1, 0]]       # không tới được cổng

# đối chiếu với oracle độc lập: BFS riêng từng phòng
def walls_oracle(rooms):
    m, n = len(rooms), len(rooms[0])
    out = [row[:] for row in rooms]
    for si in range(m):
        for sj in range(n):
            if rooms[si][sj] != INF_ROOM:
                continue
            seen = {(si, sj)}
            q = deque([(si, sj, 0)])
            while q:
                i, j, d = q.popleft()
                if rooms[i][j] == 0:
                    out[si][sj] = d
                    break
                for dx, dy in DIRS:
                    x, y = i + dx, j + dy
                    if 0 <= x < m and 0 <= y < n and rooms[x][y] != -1 and (x, y) not in seen:
                        seen.add((x, y))
                        q.append((x, y, d + 1))
    return out

rng5 = random.Random(55)
for _ in range(300):
    m, n = rng5.randint(1, 6), rng5.randint(1, 6)
    g = [[rng5.choice([-1, 0, INF_ROOM, INF_ROOM]) for _ in range(n)] for _ in range(m)]
    assert exercise_5_solution(g) == walls_oracle(g)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template BFS đa nguồn trên lưới

```python
class Solution:
    def updateMatrix(self, mat: List[List[int]]) -> List[List[int]]:
        m, n = len(mat), len(mat[0])
        ans = [[-1] * n for _ in range(m)]          # -1 = chưa thăm
        q = deque()
        for i in range(m):
            for j in range(n):
                if mat[i][j] == 0:                  # mọi nguồn cùng vào hàng đợi, d = 0
                    ans[i][j] = 0
                    q.append((i, j))
        while q:
            i, j = q.popleft()
            for dx, dy in ((-1, 0), (0, 1), (1, 0), (0, -1)):
                x, y = i + dx, j + dy
                if 0 <= x < m and 0 <= y < n and ans[x][y] == -1:
                    ans[x][y] = ans[i][j] + 1       # gán NGAY khi đưa vào hàng đợi
                    q.append((x, y))
        return ans
```

Câu thần chú: **"mọi số 0 cùng lan sóng; ô nào chạm sóng trước thì khoảng cách là số bước của sóng."**

### Checklist trước khi submit

- [ ] Ma trận kết quả tạo bằng `[[-1] * n for _ in range(m)]` (không dùng `[[-1] * n] * m`).
- [ ] Mọi ô `0` được gán `ans = 0` **và** đưa vào hàng đợi ngay từ đầu.
- [ ] Dùng `deque` + `popleft()`; đánh dấu đã thăm **lúc đưa vào**, không phải lúc lấy ra.
- [ ] Kiểm tra biên `0 <= x < m and 0 <= y < n` trước khi truy cập `ans[x][y]`.
- [ ] Test `1 x 1`, một hàng, một cột, toàn `0`, chỉ một số `0` ở góc.
- [ ] Nếu dùng DP: hai lượt quét (trên-trái rồi dưới-phải) và `INF >= m + n`.
- [ ] Skeleton đúng: `class Solution`, method `updateMatrix(self, mat: List[List[int]]) -> List[List[int]]`.